## Feature Enginnering
Preparar os dados para criação da base spec

1. Selecionar apenas variáveis validadas (excluindo entrada de features com potencial de data leakage)
2. Criar coluna previous_contact
3. Criar faixas de idade
4. Combinar indicadore de estabilidade financeira
5. Score de engajamento
6. Aplicar padronização, normalização e one hot encoding quando aplicável

### 1. Leitura dos dados

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Configura caminho para a raiz do projeto (tech_challenge_5)
PROJECT_ROOT = Path.cwd().parent

# Adiciona a raiz do projeto ao caminho de busca do Python,
# permitindo importar módulos da pasta src.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Agora o import funciona
from src.config import RAW_DATA_PATH

df = pd.read_csv(
    RAW_DATA_PATH / "bank-additional-full.csv",
    sep=";"
)

pd.set_option("display.max_columns", None) #visualizar todas as colunas
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### 2. Apaga dados com potencial de dataleakage

In [15]:
columns_to_remove = [
    "duration"
]

df = df.drop(columns=columns_to_remove)

### 3. Criação de novas variáveis para teste nos modelos

In [16]:
# ------------------------------------------------------------------
# Variável binária de contato prévio
# ------------------------------------------------------------------

df["previous_contact"] = (df["previous"] > 0).astype(int)  

df["previous_success"] = (
    df["poutcome"] == "success"
).astype(int)

# ------------------------------------------------------------------
# Variavel que agrupa faixas de idade
# ------------------------------------------------------------------

df["age_group"] = pd.cut(
    df["age"],
    bins=[0,25,35,50,65,100],
    labels=[
        "young",
        "young_adult",
        "adult",
        "senior",
        "elderly"
    ]
)

# ------------------------------------------------------------------
# variavel de risco financeiro 
# ------------------------------------------------------------------

df["financial_risk"] = (
    (df["housing"] == "yes").astype(int)
    +
    (df["loan"] == "yes").astype(int)
    +
    (df["default"] == "yes").astype(int)
)

# ------------------------------------------------------------------
# Variavel de engajamento histórico 
# ------------------------------------------------------------------

df["engagement_score"] = (
    df["campaign"]
    +
    df["previous"]
)

In [8]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,previous_contact,previous_success,age_group,financial_risk,engagement_score
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,0,senior,0,1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,0,senior,0,1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,0,adult,1,1
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,0,adult,0,1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,0,senior,1,1


4. Converte target em 0 ou 1 

In [19]:
df["y"] = df["y"].map({
    "yes": 1,
    "no": 0
})

5. Base de dados final e salvamento

In [20]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,previous_contact,previous_success,age_group,financial_risk,engagement_score
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,senior,0,1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,senior,0,1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,adult,1,1
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,adult,0,1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,senior,1,1


In [21]:
from pathlib import Path

# Caminho para a pasta data/processed
output_dir = Path("../data/processed")

# Cria a pasta caso não exista
output_dir.mkdir(parents=True, exist_ok=True)

# Salva o dataset
df.to_csv(output_dir / "bank_marketing_processed.csv", index=False)

print(f"Arquivo salvo em: {(output_dir / 'bank_marketing_processed.csv').resolve()}")

Arquivo salvo em: C:\Users\alice\OneDrive\Área de Trabalho\tech_challenge_5\data\processed\bank_marketing_processed.csv
